In [2]:
import glob
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
import yaml



In [15]:
class Get_WRF_DATA:
    """
    Class for loading, processing, and preparing WRF model outputs 
    along with static and remote sensing data for machine learning applications.
    """

    def __init__(self, start: str, end: str, path_2_yml:str):
        """
        Initialize the Get_WRF_DATA class.

        Parameters:
        -----------
        start : str
            Start year of the data (e.g., '2010').
        end : str
            End year of the data (e.g., '2015').
        """
        with open(path_2_yml, 'r') as file:
            data = yaml.safe_load(file)

        self.config = data
        
        self.start = start
        self.end = end
        
        self.dynamic_var_1 = self.config['variables']
        self.dynamic_path_1 = self.config['dynamic_path1']
        self.static_path = self.config['static_path']
        
        self.data_path_list = [self.dynamic_path_1]
        self.data_variable_list = [self.dynamic_var_1] 

    def data_split(self, path, variables):
        """
        Load WRF data across multiple years and concatenate along time.

        Parameters:
        -----------
        path : str
            Path to WRF data directory.
        variables : list
            List of variables to extract.

        Returns:
        --------
        xarray.Dataset
            Dataset containing requested variables.
        """
        assert int(self.start) <= int(self.end), 'Time step should be incremental'
    
        years = [int(self.start) + i for i in range(int(self.end) + 1 - int(self.start))]
        file_paths = []
        for yr in years:
            file_paths.extend(sorted(glob.glob(f'{path}{yr}/wrf*')))
    
        x = xr.open_mfdataset(file_paths, concat_dim='XTIME', combine='nested', parallel=True)
        return x[variables]

    def compute_water_year_cumsum(self, data):
        """
        Compute cumulative sum for each water year.

        Parameters:
        -----------
        data : np.ndarray
            4D array with shape (time, lat, lon, channels).

        Returns:
        --------
        np.ndarray
            Cumulative sum per water year.
        """
        start_year = int(self.start)
        end_year = int(self.end)

        def is_leap_year(year):
            return (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0)

        water_year_lengths = [366 if is_leap_year(year) else 365 for year in range(start_year, end_year + 1)]
        start_indices = np.cumsum([0] + water_year_lengths[:-1])
        end_indices = np.cumsum(water_year_lengths)

        cumsum_result = np.zeros_like(data)
        for start, end in zip(start_indices, end_indices):
            cumsum_result[start:end] = np.cumsum(data[start:end], axis=0)

        return cumsum_result

    def merge_wrf_features_matrix(self):
        """
        Wrapper to extract and merge WRF feature matrices.

        Returns:
        --------
        xarray.Dataset
            Dataset containing selected WRF variables.
        """
        for path, variables in zip(self.data_path_list, self.data_variable_list):
            return self.data_split(path, variables)

    def wrf_variables(self):
        """
        Load and process WRF dynamic variables, including accumulation for PRCP and SNOWNC.

        Returns:
        --------
        np.ndarray
            4D array (time, lat, lon, channels).
        """
        dataset = self.data_split(self.dynamic_path_1, self.dynamic_var_1)

        X = np.empty(shape=[dataset.XTIME.size, 
                            dataset.south_north.size, 
                            dataset.west_east.size, 
                            len(self.dynamic_var_1) + 2],
                     dtype=np.float64)
        
        for var_index, var in enumerate(self.dynamic_var_1):
            if var == 'PRCP':
                for ti in range(dataset.XTIME.size): 
                    ti_data = dataset[var].isel(XTIME=ti).values
                    if np.min(ti_data) < 0:
                        print(f'Error of {var} min at time {ti}')
                        ti_data = ti_data * 0
                    X[ti, :, :, var_index] = ti_data

            elif var == 'SNOWNC':
                accum_snownc = dataset[var].values
                deaccumulate_snownc = np.zeros_like(accum_snownc)
                deaccumulate_snownc[1:] = np.diff(accum_snownc, axis=0)
                for ti in range(dataset.XTIME.size):
                    ti_data = deaccumulate_snownc[ti]
                    if np.min(ti_data) < 0:
                        print(f'Error of {var} min at time {ti}')
                        ti_data = ti_data * 0
                    X[ti, :, :, var_index] = ti_data

            else:
                for ti in range(dataset.XTIME.size):
                    X[ti, :, :, var_index] = dataset[var].isel(XTIME=ti).values

        # Add accumulated PRCP and SNOWNC
        X[:, :, :, var_index + 1] = self.compute_water_year_cumsum(X[:, :, :, 0])
        X[:, :, :, var_index + 2] = self.compute_water_year_cumsum(X[:, :, :, 1])

        return X

    def merge_static(self, write_to_netcdf = False):
        """
        Add static elevation data to dynamic features.

        Returns:
        --------
        np.ndarray
            Feature matrix with static elevation as a channel.
        """
        d = self.wrf_variables()
        NT = d.shape[0]
        static = xr.open_dataset(self.static_path)['HGT'].isel(Time=0).values

        static_data = np.repeat(static[np.newaxis, :, :, np.newaxis], NT, axis=0)
        return np.concatenate([d, static_data], axis=-1)


    def merge_of_year_encoding(self, write_to_netcdf=True, output_path="wrf_forcings.nc"):
        """
        Add day-of-year sine and cosine encoding to the feature matrix and optionally save to NetCDF.
    
        Parameters:
        -----------
        write_to_netcdf : bool
            Whether to save the final encoded dataset to a NetCDF file.
        output_path : str
            File path to save the NetCDF output (if write_to_netcdf is True).
    
        Returns:
        --------
        np.ndarray
            Final feature matrix with temporal encoding.
        """
        from datetime import datetime, timedelta
    
        start1 = f'{int(self.start) - 1}-10-01'
        end1 = f'{int(self.end)}-09-30'
        start = tuple(map(int, start1.split('-')))
        end = tuple(map(int, end1.split('-')))
    
        data = self.merge_static()

        def is_leap_year(year):
            return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
    
        def day_of_year_with_leap(date):
            return (date - datetime(date.year, 1, 1)).days + 1
    
        start_date = datetime(*start)
        end_date = datetime(*end)
        num_days = (end_date - start_date).days + 1
    
        # Time coordinate with datetime objects
        dates = np.array([start_date + timedelta(days=i) for i in range(num_days)])
        days_of_year = np.array([day_of_year_with_leap(d) for d in dates])
        years = np.array([d.year for d in dates])
        is_leap = np.array([is_leap_year(y) for y in years])
    
        day_sin = np.sin(2 * np.pi * days_of_year / np.where(is_leap, 366, 365))
        day_cos = np.cos(2 * np.pi * days_of_year / np.where(is_leap, 366, 365))
    
        day_sin = day_sin[:, np.newaxis, np.newaxis, np.newaxis]
        day_cos = day_cos[:, np.newaxis, np.newaxis, np.newaxis]
    
        day_sin = np.repeat(day_sin, data.shape[1], axis=1)
        day_sin = np.repeat(day_sin, data.shape[2], axis=2)
        day_cos = np.repeat(day_cos, data.shape[1], axis=1)
        day_cos = np.repeat(day_cos, data.shape[2], axis=2)
    
        final_data = np.concatenate([data, day_sin, day_cos], axis=-1)
    
        if write_to_netcdf:
            base_var_names = self.dynamic_var_1 + ['PRCP_CUMSUM', 'SNOWNC_CUMSUM', 'ELEVATION']
            var_names = base_var_names + ['DAY_SIN', 'DAY_COS']
            NT, NY, NX, _ = final_data.shape
    
            dataset = xr.Dataset(
                {
                    var: (["XTIME", "south_north", "west_east"], final_data[:, :, :, i])
                    for i, var in enumerate(var_names)
                },
                coords={
                    "XTIME": dates,  # datetime array here
                    "south_north": np.arange(NY),
                    "west_east": np.arange(NX),
                }
            )
            # leave out the daily precip (prcp and snownc) values, save only the incremental
            subset = ['PRCP_CUMSUM', 'SNOWNC_CUMSUM', 'TMIN', 'TMAX', 'ELEVATION', 'DAY_SIN','DAY_COS']
            dataset[subset].to_netcdf(output_path)
            print(f"Saved final encoded dataset to {output_path}")
    
        return final_data


    def write_feature_matrix_to_npy(self):
        """
        Save final feature matrix to .npy file.

        Parameters:
        -----------
        path : str
            File path to save the .npy file.
        """
        data = self.merge_of_year_encoding()
        try:
            np.save(self.config['feat_mat_write_path'], data)
            print(f"Array successfully saved to {self.config['feat_mat_write_path']}")
        except Exception as e:
            print(f"An error occurred while saving the array: {e}")

    def get_target_vector(self):
        """
        Load target variable (e.g., SWE).

        Returns:
        --------
        np.ndarray
            4D array (time, lat, lon, 1) with the target variable.
        """
        dataset = self.data_split(self.dynamic_path_1, 'SNOW')
        X = np.empty(shape=[dataset.XTIME.size, 
                            dataset.south_north.size, 
                            dataset.west_east.size, 
                            1],
                     dtype=np.float64)
        X[:, :, :, 0] = dataset.values
        return X

    def write_target_vector_to_npy(self):
        """
        Save target variable array to .npy file.

        Parameters:
        -----------
        path : str
            File path to save the .npy file.
        """
        data = self.get_target_vector()
        try:
            np.save(self.config['feat_vec_write_path'], data)
            print(f"Array successfully saved to {self.config['feat_vec_write_path']}")
        except Exception as e:
            print(f"An error occurred while saving the array: {e}")

In [16]:
WRF_data = Get_WRF_DATA(2003,2017,'config.yml')

In [47]:
# wrf_forcings = WRF_data.merge_of_year_encoding()

In [32]:
# target = WRF_data.data_split('/bsuscratch/stanleyakor/uppercolorado/WY','SNOW')

In [33]:
# target.to_netcdf('../data/swe.nc')